
Identity Resolution Example using Vector Embeddings from a Doc2Vec model

David C. Sisk, 2026-03-03

This notebook requires a pre-trained doc2vec model, which you can download from here (quite large):
https://mega.nz/file/S2wxxArK#akoKdYl4SaO2JX29AHDWQ8skbDGnxrih9-mViK_ezWA

In [ ]:
#! pip install pandas
#! pip install scikit-learn
#! pip install gensim

In [50]:
import pandas as pd

df_inputdata = pd.read_csv('sample-data-messy_200.csv')

df_inputdata.shape

(200, 6)

In [51]:
# Calculate vector embeddings using a pretrained Doc2Vec model instead of an LLM
from gensim.models.doc2vec import Doc2Vec

# Load the pretrained doc2vec model
doc2vec_model = Doc2Vec.load("doc2vec_wikipedia_dm.model")

# Function to generate embeddings for a row
def generate_doc2vec_embedding(row):
    # Exclude 'row_id' and 'true_id' columns and concatenate other columns
    text = " ".join(str(value) for key, value in row.items() if key not in ['row_id', 'true_id'])
    # Infer vector using the doc2vec model
    vector = doc2vec_model.infer_vector(text.split())
    return vector

# Apply the embedding function to each row
df_inputdata['embedding'] = df_inputdata.apply(generate_doc2vec_embedding, axis=1)

print("df_inputdata.shape:", df_inputdata.shape)

embedding_dimension = len(df_inputdata['embedding'].iloc[0])
print(f"The embedding dimension count is: {embedding_dimension}")

df_inputdata.shape: (200, 7)
The embedding dimension count is: 200


In [52]:
# L2-normalize vector embeddings and store in a new column
target_df = df if ("df" in globals() and "embedding" in df.columns) else df_inputdata

target_df["l2n_embedding"] = target_df["embedding"].apply(
    lambda v: v / np.linalg.norm(v) if np.linalg.norm(v) != 0 else v
)

In [53]:
# Using the inputdata above, construct a pairwise dataframe for every unique combination
from itertools import combinations

# Create lists of row_id and true_id
row_ids = df_inputdata['row_id'].tolist()
true_ids = df_inputdata['true_id'].tolist()

# Helper function to get field value, replacing NaN with empty string
def get_field(row, col):
    val = df_inputdata.iloc[row][col]
    return '' if pd.isna(val) else str(val)

# Create all pairwise combinations
pairwise_data = []
for i in range(len(df_inputdata)):
    for j in range(len(df_inputdata)):
        if i != j:  # Don't pair a row with itself
            # Concatenate name, email, address, and phone for both rows
            data1 = ' | '.join([
                get_field(i, 'name'),
                get_field(i, 'email'),
                get_field(i, 'address'),
                get_field(i, 'phone')
            ])
            data2 = ' | '.join([
                get_field(j, 'name'),
                get_field(j, 'email'),
                get_field(j, 'address'),
                get_field(j, 'phone')
            ])
            pairwise_data.append({
                'row_id1': row_ids[i],
                'true_id1': true_ids[i],
                'data1': data1,
                'embedding1': df_inputdata['l2n_embedding'].iloc[i],
                'row_id2': row_ids[j],
                'true_id2': true_ids[j],
                'data2': data2,
                'embedding2': df_inputdata['l2n_embedding'].iloc[j]
            })

df_pairwise = pd.DataFrame(pairwise_data)

df_pairwise.shape

(39800, 8)

In [79]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity,score,match
7529,row-0060,id-0064,Asutin Ward | austin.ward0@example.net | 6465 ...,"[-0.043401375, -0.10754305, 0.07539765, 0.0611...",row-0255,id-0050,Araon Sanchez | aaron.sanchez0@example.net | 5...,"[-0.11556514, -0.07524866, 0.06440097, -0.1091...",-0.069003,15.98,0
13808,row-0114,id-0064,"Ward, Austin | award@mail.example.org | 6464 6...","[-0.070006, -0.009873286, -0.07822911, -0.0075...",row-0134,id-0028,"Logan King | | 28 Brook Blvd, Springfield, IL...","[-0.022219755, -0.009978506, -0.015017653, 0.1...",-0.018768,20.11,0
14727,row-0125,id-0094,"Stone, Megan | mstone@mail.example.org | 9494 ...","[0.10876613, 0.07024447, -0.020779096, 0.05682...",row-0003,id-0013,"James Harris | | 13 Sycamore St, Springfield,...","[0.01963692, -0.014016454, -0.0061177695, 0.07...",-0.053359,17.27,0


In [55]:
# For each pairwise row, calculate cosine similarity between embedding1 and
# embedding2, and store them in a new column called "cosine_similarity"
import numpy as np

def cosine_sim(v1, v2):
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    return float(np.dot(v1, v2) / denom) if denom != 0 else np.nan

df_pairwise["cosine_similarity"] = [
    cosine_sim(v1, v2)
    for v1, v2 in zip(df_pairwise["embedding1"], df_pairwise["embedding2"])
]


In [56]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity
1178,row-0009,id-0007,"Isabella Moore | | 7 Walnut Ave., Springfield...","[0.009479984, -0.08805479, -0.029701982, 0.088...",row-0277,id-0002,Oilvia Smith | olivia.smith1@example.net | 235...,"[-0.015969375, 0.014748556, -0.028459117, -0.0...",-0.000816
5188,row-0041,id-0085,"Hailey Webb | | 85 Northview St, Springfield,...","[0.02662937, -0.023364812, -0.067288995, 0.122...",row-0023,id-0075,"Pierce, Stella | spierce@mail.example.org | 75...","[-0.077609986, -0.12569118, 0.04789548, 0.0062...",0.009217
34021,row-0258,id-0014,Benjamin Martin | | | (555) 010-0014,"[0.028801654, -0.106829114, -0.007789485, 0.10...",row-0289,id-0006,Spohia Wilson | sophia.wilson5@example.net | 6...,"[-0.0021655143, -0.028640008, -0.028909711, 0....",0.613622


In [57]:
# Min-max normalize cosine similarity to a 0-100 scale and store that as the score
min_sim = df_pairwise["cosine_similarity"].min()
max_sim = df_pairwise["cosine_similarity"].max()

if max_sim == min_sim:
    df_pairwise["score"] = 100.0
else:
    df_pairwise["score"] = (
        (df_pairwise["cosine_similarity"] - min_sim) / (max_sim - min_sim) * 100
    )

# round score to 2 decimal places
df_pairwise["score"] = df_pairwise["score"].round(2)

In [58]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity,score
27521,row-0218,id-0040,"Nathan Perez | | 40 Grove Blvd, Springfield, ...","[0.02593504, -0.031810623, -0.011864088, 0.137...",row-0097,id-0031,"Sebastian Hill | | 31 Lake Rd., Springfield, ...","[0.017826417, -0.0647667, -0.016813407, 0.1105...",0.882918,94.19
1061,row-0009,id-0007,"Isabella Moore | | 7 Walnut Ave., Springfield...","[0.009479984, -0.08805479, -0.029701982, 0.088...",row-0112,id-0042,"Isaac Turner | | 42 Crest Ave., Springfield, ...","[0.03651542, -0.07571233, -0.040156387, 0.0794...",0.902855,95.82
2278,row-0019,id-0053,"Reed, Adrian | areed@mail.example.org | 5353 5...","[-0.03227577, 0.09330588, 0.11810903, 0.104003...",row-0155,id-0068,Cloe Bryant | cole.bryant4@example.net | 6869 ...,"[0.08579744, 0.062446047, -0.004551166, -0.072...",0.052553,25.97


In [59]:
# Display rows where true_id1 equals true_id2
df_pairwise[df_pairwise['true_id1'] == df_pairwise['true_id2']]


,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity,score
142,row-0002,id-0080,"Chavez, Leah | lchavez@mail.example.org | 8080...","[-0.08302811, 0.10027081, -0.051157672, 0.0529...",row-0224,id-0080,Laeh Chavez | leah.chavez2@example.net | 8081 ...,"[0.046830144, -0.107680865, -0.09811691, -0.04...",-0.050130,17.53
313,row-0003,id-0013,"James Harris | | 13 Sycamore St, Springfield,...","[0.01963692, -0.014016454, -0.0061177695, 0.07...",row-0190,id-0013,Jmaes Harris | james.harris5@example.net | 131...,"[-0.1056912, 0.009121831, -0.11304917, -0.0141...",0.003519,21.94
486,row-0004,id-0030,"Lopez, Levi | llopez@mail.example.org | 3030 3...","[0.11148599, -0.012908586, 0.025418185, 0.0562...",row-0154,id-0030,Lvei Lopez | levi.lopez1@example.net | 3031 30...,"[-0.03872467, -0.09728488, -0.0947527, -0.0758...",0.089211,28.98
647,row-0005,id-0011,"Jackson, Henry | hjackson@mail.example.org | 1...","[-0.12339445, 0.03226886, -0.10517143, -0.1236...",row-0087,id-0011,"Henry Jackson | | 11 Willow Dr, Springfield, ...","[0.06468036, -0.036838863, -0.061384473, 0.107...",0.017338,23.08
868,row-0008,id-0041,Rayn Roberts | ryan.roberts5@example.net | | ...,"[0.02738294, -0.0014345669, -0.050077375, 0.09...",row-0121,id-0041,"Roberts, Ryan | rroberts@mail.example.org | 41...","[0.11935053, -0.07578898, -0.0038233218, 0.041...",0.030924,24.19
...,...,...,...,...,...,...,...,...,...,...
38954,row-0294,id-0018,"Alexander Robinson | | 18 Cypress Ave., Sprin...","[0.018222492, -0.06894328, -0.03863604, 0.0707...",row-0232,id-0018,"Robinson, Alexander | arobinson@mail.example.o...","[-0.034148503, -0.03986555, -0.051498003, -0.0...",0.025133,23.72
39159,row-0296,id-0039,Claeb Mitchell | caleb.mitchell3@example.net |...,"[-0.07634832, 0.10938314, -0.048521478, 0.1059...",row-0239,id-0039,"Mitchell, Caleb | cmitchell@mail.example.org |...","[-0.019590054, 0.008616869, -0.05270591, 0.053...",0.140869,33.22
39261,row-0298,id-0047,"Edwards, Thomas | tedwards@mail.example.org | ...","[0.036823887, -0.05225331, -0.018481566, 0.090...",row-0096,id-0047,Tohmas Edwards | thomas.edwards4@example.net |...,"[0.105478294, 0.11755969, 0.06930039, 0.113695...",-0.033051,18.94
39531,row-0299,id-0005,Aav Miller | ava.miller4@example.net | 566 5 B...,"[-0.1168034, -0.08267889, -0.1141718, 0.110367...",row-0207,id-0005,"Miller, Ava | amiller@mail.example.org | 567 5...","[0.0046180477, 0.024925362, -0.041047614, -0.0...",0.032609,24.33


In [60]:
# Examine the score spread where they were true matches

# Filter rows where true_id1 equals true_id2
matches = df_pairwise[df_pairwise['true_id1'] == df_pairwise['true_id2']]

# Calculate statistics
count = len(matches)
min_score = matches['score'].min()
avg_score = matches['score'].mean()
max_score = matches['score'].max()

print(f"Count: {count}")
print(f"Min Score: {min_score}")
print(f"Avg Score: {avg_score:.2f}")
print(f"Max Score: {max_score}")

Count: 200
Min Score: 5.36
Avg Score: 31.48
Max Score: 97.9


In [61]:
# Examine the score spread where they are NOT true matches

# Filter rows where true_id1 does NOT equal true_id2
non_matches = df_pairwise[df_pairwise['true_id1'] != df_pairwise['true_id2']]

# Calculate statistics
count = len(non_matches)
min_score = non_matches['score'].min()
avg_score = non_matches['score'].mean()
max_score = non_matches['score'].max()

print(f"Count: {count}")
print(f"Min Score: {min_score}")
print(f"Avg Score: {avg_score:.2f}")
print(f"Max Score: {max_score}")

Count: 39600
Min Score: 0.0
Avg Score: 38.02
Max Score: 100.0


In [80]:
# Choose the matching score cutoff threshold that gets the most acceptable mix 
# of false positives and false negatives
#cutoff = 47.76  # This is the middle of the average scores...reasonable starting point
cutoff = 85.00
df_pairwise["match"] = (df_pairwise["score"] >= cutoff).astype(int)

# show counts of match values (0 = non-match, 1 = match)
print("Predicted matches:", (df_pairwise["match"] == 1).sum(), 
      "...predicted non-matches:", (df_pairwise["match"] == 0).sum())

# Calculate Precision & Recall as our accuracy metrics
# True positives: predicted match (1) and actually same person (true_id1 == true_id2)
true_positives = df_pairwise[
    (df_pairwise["match"] == 1) & (df_pairwise["true_id1"] == df_pairwise["true_id2"])
]

# False positives: predicted match (1) but actually different people (true_id1 != true_id2)
false_positives = df_pairwise[
    (df_pairwise["match"] == 1) & (df_pairwise["true_id1"] != df_pairwise["true_id2"])
]

# False negatives: predicted no match (0) but actually same person (true_id1 == true_id2)
false_negatives = df_pairwise[
    (df_pairwise["match"] == 0) & (df_pairwise["true_id1"] == df_pairwise["true_id2"])
]

# Calculate Precision & Recall, plus F1 score
precision = len(true_positives) / (len(true_positives) + len(false_positives))
recall = len(true_positives) / (len(true_positives) + len(false_negatives))
F1_score = 2 * (precision * recall) / (precision + recall)

print(f"PRECISION: {precision:.4f}  (How many predicted matches were correct true matches?)")
print(f"RECALL: {recall:.4f}  (How many true matches were correctly predicted?)")
print(f"F1 SCORE: {F1_score:.4f}  (Harmonic mean of precision and recall)") 

Predicted matches: 7128 ...predicted non-matches: 32672
PRECISION: 0.0036  (How many predicted matches were correct true matches?)
RECALL: 0.1300  (How many true matches were correctly predicted?)
F1 SCORE: 0.0071  (Harmonic mean of precision and recall)
